# Machine Learning over Structured Data — Benchmark and Baselines

**Running example:** RelBench `rel-f1`, task `driver-position`
*For each driver who raced in the past year, predict their average finishing position over the next 60 days.*

Outline
1. Load a relational database and look at its tables, keys and time columns
2. Load a prediction task and its temporal train / val / test split
3. Baselines: global mean and **per-driver historical mean** (hands-on)
4. Flatten the database into one feature table (no leakage!)
5. Demo: TabPFN on the flat table
6. Compare results

In [1]:
%pip install -q -U "relbench>=3.0"

Note: you may need to restart the kernel to use updated packages.


## 1. The database

In [2]:
from pathlib import Path
import pandas as pd
from relbench.base import Table, Database

raw_dir = Path("./../oulad")

# 1. Read CSV files
courses_df = pd.read_csv(raw_dir / "courses.csv")
assessments_df = pd.read_csv(raw_dir / "assessments.csv")
vle_df = pd.read_csv(raw_dir / "vle.csv")
student_info_df = pd.read_csv(raw_dir / "studentInfo.csv")
student_reg_df = pd.read_csv(raw_dir / "studentRegistration.csv")
student_assess_df = pd.read_csv(raw_dir / "studentAssessment.csv")
student_vle_df = pd.read_csv(raw_dir / "studentVle.csv")

# 2. Build surrogate single-column keys for composite relationships
courses_df["course_id"] = (
    courses_df["code_module"] + "_" + courses_df["code_presentation"]
)
assessments_df["course_id"] = (
    assessments_df["code_module"] + "_" + assessments_df["code_presentation"]
)
vle_df["course_id"] = (
    vle_df["code_module"] + "_" + vle_df["code_presentation"]
)

student_info_df["course_id"] = (
    student_info_df["code_module"] + "_" + student_info_df["code_presentation"]
)

student_info_df["student_reg_id"] = (
    student_info_df["course_id"] + "_" + student_info_df["id_student"].astype(str)
)

student_reg_df["course_id"] = (
    student_reg_df["code_module"] + "_" + student_reg_df["code_presentation"]
)

student_reg_df["student_reg_id"] = (
    student_reg_df["course_id"] + "_" + student_reg_df["id_student"].astype(str)
)

student_vle_df["student_reg_id"] = (
    student_vle_df["code_module"]
    + "_"
    + student_vle_df["code_presentation"]
    + "_"
    + student_vle_df["id_student"].astype(str)
)

student_assess_df["student_asseigment_id"] = (
    student_assess_df["id_assessment"].astype(str) + "_" + student_assess_df["id_student"].astype(str)
)

# 3. Create RelBench Table objects
table_dict = {
    "courses": Table(
        df=courses_df,
        fkey_col_to_pkey_table={},
        pkey_col="course_id",
    ),
    "assessments": Table(
        df=assessments_df,
        fkey_col_to_pkey_table={"course_id": "courses"},
        pkey_col="id_assessment",
    ),
    "vle": Table(
        df=vle_df,
        fkey_col_to_pkey_table={"course_id": "courses"},
        pkey_col="id_site",
    ),
    "studentInfo": Table(
        df=student_info_df,
        fkey_col_to_pkey_table={"course_id": "courses"},
        pkey_col="student_reg_id",
    ),
    "studentRegistration": Table(
        df=student_reg_df,
        fkey_col_to_pkey_table={"student_reg_id": "studentInfo"},
        pkey_col="student_reg_id",
    ),
    "studentAssessment": Table(
        df=student_assess_df,
        fkey_col_to_pkey_table={"id_assessment": "assessments"},
        pkey_col="student_asseigment_id",
    ),
    "studentVle": Table(
        df=student_vle_df,
        fkey_col_to_pkey_table={"id_site": "vle", "student_id": "studentInfo"},
        pkey_col=None,
    ),
}

# 4. Construct the Database
db = Database(table_dict=table_dict)
print(db)

Database(courses=22, assessments=206, vle=6364, studentInfo=32593, studentRegistration=32593, studentAssessment=173912, studentVle=10655280)


In [3]:
import pandas as pd

print("=" * 85)
print(f"{'DATABASE SCHEMA & DATATYPES (FROM NOTEBOOK db)':^85}")
print("=" * 85)

for name, table in db.table_dict.items():
    print(f"\nTABLE: {name}")
    print(f"Rows        : {len(table.df):,}")
    print(f"Primary Key : {table.pkey_col}")
    print(f"Foreign Keys: {table.fkey_col_to_pkey_table or 'None'}")
    print(f"Time Column : {table.time_col or 'None'}")

    # Build column summary with datatypes, key role, null count, and sample value
    col_summary = []
    for col in table.df.columns:
        if col == table.pkey_col:
            key_role = "PK"
        elif col in table.fkey_col_to_pkey_table:
            key_role = f"FK -> {table.fkey_col_to_pkey_table[col]}"
        else:
            key_role = "-"

        sample_val = table.df[col].dropna().iloc[0] if not table.df[col].dropna().empty else "None"

        col_summary.append({
            "Column": col,
            "Data Type": str(table.df[col].dtype),
            "Key Role": key_role,
            "Null Count": table.df[col].isna().sum(),
            "Sample Value": str(sample_val)[:30]
        })

    summary_df = pd.DataFrame(col_summary)
    print(summary_df.to_string(index=False))
    print("-" * 85)

                   DATABASE SCHEMA & DATATYPES (FROM NOTEBOOK db)                    

TABLE: courses
Rows        : 22
Primary Key : course_id
Foreign Keys: None
Time Column : None
                    Column Data Type Key Role  Null Count Sample Value
               code_module       str        -           0          AAA
         code_presentation       str        -           0        2013J
module_presentation_length     int64        -           0          268
                 course_id       str       PK           0    AAA_2013J
-------------------------------------------------------------------------------------

TABLE: assessments
Rows        : 206
Primary Key : id_assessment
Foreign Keys: {'course_id': 'courses'}
Time Column : None
           Column Data Type      Key Role  Null Count Sample Value
      code_module       str             -           0          AAA
code_presentation       str             -           0        2013J
    id_assessment     int64            PK           0

In [4]:
import numpy as np
import pandas as pd
import relbench

dataset = relbench.load_dataset("rel-f1")
db = dataset.get_db()          # rows after the test timestamp are hidden
print("val timestamp: ", dataset.val_timestamp)
print("test timestamp:", dataset.test_timestamp)

C:\Users\Roy\dev\struct-ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


val timestamp:  2005-01-01 00:00:00
test timestamp: 2010-01-01 00:00:00


In [5]:
# Every table: size, primary key, foreign keys (column -> referenced table), time column
for name, table in db.table_dict.items():
    print(f"{name:22s} rows={len(table.df):6d}  pk={table.pkey_col!s:24s} "
          f"fks={table.fkey_col_to_pkey_table}  time={table.time_col}")

races                  rows=   820  pk=raceId                   fks={'circuitId': 'circuits'}  time=date
qualifying             rows=  4082  pk=qualifyId                fks={'raceId': 'races', 'driverId': 'drivers', 'constructorId': 'constructors'}  time=date
constructor_standings  rows= 10170  pk=constructorStandingsId   fks={'raceId': 'races', 'constructorId': 'constructors'}  time=date
standings              rows= 28115  pk=driverStandingsId        fks={'raceId': 'races', 'driverId': 'drivers'}  time=date
constructors           rows=   211  pk=constructorId            fks={}  time=None
drivers                rows=   857  pk=driverId                 fks={}  time=None
constructor_results    rows=  9408  pk=constructorResultsId     fks={'raceId': 'races', 'constructorId': 'constructors'}  time=date
circuits               rows=    77  pk=circuitId                fks={}  time=None
results                rows= 20323  pk=resultId                 fks={'raceId': 'races', 'driverId': 'drivers

In [6]:
drivers = db.table_dict["drivers"].df
results = db.table_dict["results"].df
races = db.table_dict["races"].df
results.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionOrder,points,laps,milliseconds,fastestLap,rank,statusId,date
0,0,0,660,152,18.0,21,11.0,11,0.0,64,NaN,NaN,NaN,16,1950-05-13
1,1,0,790,149,8.0,12,NaN,21,0.0,2,NaN,NaN,NaN,126,1950-05-13
2,2,0,579,49,1.0,3,NaN,12,0.0,62,NaN,NaN,NaN,44,1950-05-13
3,3,0,661,149,9.0,10,NaN,20,0.0,5,NaN,NaN,NaN,6,1950-05-13
4,4,0,789,152,17.0,7,NaN,19,0.0,8,NaN,NaN,NaN,51,1950-05-13


**Try it:** follow one row of `results` through its foreign keys. Which driver, which race, which circuit?

In [7]:
row = results.iloc[0]
race = races.set_index("raceId").loc[row["raceId"]]
driver = drivers.set_index("driverId").loc[row["driverId"]]
print(driver["forename"], driver["surname"], "finished", row["positionOrder"], "in", race["name"], race["year"])

Johnny Claes finished 11 in British Grand Prix 1950


## 2. The task

In [8]:
task = dataset.load_task("driver-position") # For each driver who raced in the past year, predict their average finishing position over the next 60 days.
print("type:", task.task_type, "| entity:", task.entity_col, "| target:", task.target_col,
      "| horizon:", task.timedelta)

train_df = task.get_table("train").df
val_df = task.get_table("val").df
test_df = task.get_table("test").df      # target column is hidden

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5s} {df.shape}  {df['date'].min().date()} -> {df['date'].max().date()}")
train_df.head()

type: TaskType.REGRESSION | entity: driverId | target: position | horizon: 60 days 00:00:00
train (7453, 3)  1950-06-19 -> 2004-09-03
val   (499, 3)  2005-03-02 -> 2009-10-07
test  (760, 2)  2010-03-02 -> 2016-05-29


,date,driverId,position
0,1950-06-19,498,18.0
1,1950-06-19,579,1.0
2,1950-06-19,589,15.0
3,1950-06-19,626,4.0
4,1950-06-19,627,9.0


Each row is *(driverId, date, position)*: "standing at `date`, what will this driver's average finishing position be over the next 60 days?"

The split is **temporal**: all train dates < val dates < test dates.

RelBench scores regression with **NMAE** = MAE / (std of the train target). We also print plain MAE (in finishing positions), which is easier to interpret.

In [9]:
from sklearn.metrics import mean_absolute_error

val_table = task.get_table("val")
results_table = {}

def report(name, pred):
    """Evaluate predictions for the val rows and store the result."""
    pred = np.asarray(pred, dtype=float)
    nmae = task.evaluate(pred, val_table)["nmae"]
    mae = mean_absolute_error(val_df["position"], pred)
    results_table[name] = {"val MAE": mae, "val NMAE": nmae}
    print(f"{name:25s} MAE={mae:.3f}  NMAE={nmae:.3f}")

## 3. Baselines

### 3a. Global mean
Predict the same number for everyone: the mean of the training targets.

In [10]:
global_mean = train_df["position"].mean()
pred_global = np.full(len(val_df), global_mean)
report("global mean", pred_global)

global mean               MAE=4.334  NMAE=0.617


### 3b. Per-driver historical mean (hands-on)

$$\hat{y}_{d} = \frac{1}{|H_d|} \sum_{t \in H_d} y_{d,t}$$

where $H_d$ are driver $d$'s **training** rows. All of them are dated before the validation timestamp, so this uses no future information.

In [11]:
# EXERCISE: per-driver historical mean
# 1. For every driver, compute the mean `position` over the TRAIN table only.
# 2. Map those means onto the validation rows (by driverId).
# 3. Drivers that never appear in train -> fall back to `global_mean`.

driver_means = ...   # TODO: a Series indexed by driverId
pred_entity = ...    # TODO: one prediction per row of val_df

report("entity mean", pred_entity)

TypeError: float() argument must be a string or a real number, not 'ellipsis'

In [ ]:
# STRETCH (optional): use only each driver's last k training observations.
# Does a "recent" mean beat the all-time mean? Why might it?

## 4. Flattening: one row per (driver, date)

TabPFN needs a single table. For each row of the task table we aggregate the driver's **past** results, meaning only races strictly before `date`. Using all of `results` would leak the future.

In [ ]:
import duckdb

FEATURE_SQL = """
SELECT
    t.driverId,
    t.date,
    COUNT(r.resultId)                                   AS n_races,
    AVG(r.positionOrder)                                AS mean_finish,
    AVG(r.grid)                                         AS mean_grid,
    SUM(r.points)                                       AS total_points,
    AVG(CASE WHEN r.date > t.date - INTERVAL 1 YEAR
             THEN r.positionOrder END)                  AS mean_finish_last_year,
    -- statusId 1 = "Finished" (same DNF definition as RelBench's driver-dnf task)
    AVG(CASE WHEN r.statusId != 1 THEN 1.0 ELSE 0 END)  AS dnf_rate,
    DATE_DIFF('day', MAX(r.date), t.date)               AS days_since_last_race,
    DATE_DIFF('day', ANY_VALUE(d.dob), t.date) / 365.25 AS age
FROM task_rows t
JOIN drivers d ON d.driverId = t.driverId
LEFT JOIN results r
       ON r.driverId = t.driverId
      AND r.date < t.date            -- only the past!
GROUP BY t.driverId, t.date
"""

def make_features(task_df):
    con = duckdb.connect()
    con.register("task_rows", task_df[["driverId", "date"]])
    con.register("drivers", drivers)
    con.register("results", results)
    feats = con.sql(FEATURE_SQL).df()
    con.close()
    # keep the task table's row order
    return task_df[["driverId", "date"]].merge(feats, on=["driverId", "date"], how="left")

X_train_full = make_features(train_df)
X_val_full = make_features(val_df)
FEATURES = [c for c in X_train_full.columns if c not in ("driverId", "date")]
X_train_full.head()

**Checkpoint:** why `r.date < t.date` and not just "all results of the driver"?
What would happen to the validation score if we removed that condition?

## 5. Demo: TabPFN

TabPFN is a transformer **pre-trained on millions of synthetic tables**. `fit` stores the training rows; `predict` runs one forward pass with the training rows as context ("in-context learning").

* GPU recommended. On CPU, keep ≤ ~5,000 training rows.
* The model weights download on first use and may require accepting a license on Hugging Face. **Do this before class.**

In [ ]:
!pip install -q tabpfn

In [ ]:
import torch
from tabpfn import TabPFNRegressor

MAX_TRAIN = 10_000 if torch.cuda.is_available() else 5_000
train_sub = (X_train_full.assign(y=train_df["position"].to_numpy())
             .sort_values("date").tail(MAX_TRAIN))      # most recent rows
print("training rows used:", len(train_sub), "| device:", "cuda" if torch.cuda.is_available() else "cpu")

reg = TabPFNRegressor()
reg.fit(train_sub[FEATURES], train_sub["y"])
pred_tabpfn = reg.predict(X_val_full[FEATURES])
report("TabPFN (flat features)", pred_tabpfn)

**Ablation:** is the gain from the model, or from the features?
Give TabPFN only a single feature, `mean_finish` (≈ the historical mean), and compare.

In [ ]:
reg_1f = TabPFNRegressor()
reg_1f.fit(train_sub[["mean_finish"]], train_sub["y"])
report("TabPFN (mean_finish only)", reg_1f.predict(X_val_full[["mean_finish"]]))

## 6. Compare

In [ ]:
pd.DataFrame(results_table).T.sort_values("val MAE")

Discussion
1. Does TabPFN beat the per-driver mean? By how much?
2. Is the improvement due to the model or to the features we gave it?
3. Which other tables (`qualifying`, `standings`, `constructors`, ...) could add signal, and how would you keep them leak-free?